In [0]:
# ============================================
# GOLD LAYER - DIM MERCHANT (EVENT-TIME SCD2)
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

df_silver = spark.table("silver.transactions_clean")

# Normalize date
df_silver = df_silver.withColumn(
    "transaction_date",
    to_date("date")
)


In [0]:
# Detect merchant versions from transaction history
merchant_versions = (
    df_silver
    .groupBy(
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "mcc"
    )
    .agg(
        min("transaction_date").alias("valid_from")
    )
)


In [0]:
# Calculate valid_to
window_spec = Window.partitionBy("merchant_id").orderBy("valid_from")

merchant_versions = (
    merchant_versions
    .withColumn(
        "next_valid_from",
        lead("valid_from").over(window_spec)
    )
    .withColumn(
        "valid_to",
        expr("date_sub(next_valid_from, 1)")
    )
    .withColumn(
        "is_current",
        when(col("next_valid_from").isNull(), True).otherwise(False)
    )
    .drop("next_valid_from")
)


In [0]:
# Add surrogate key
dim_merchant = (
    merchant_versions
    .withColumn("merchant_sk", monotonically_increasing_id())
)


In [0]:
# Write table
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

dim_merchant.write.mode("overwrite").format("delta").saveAsTable(
    "gold.dim_merchant"
)


In [0]:
display(spark.table("gold.dim_merchant").filter("merchant_id = 100"))

merchant_id,merchant_city,merchant_state,mcc,merchant_sk,valid_from,valid_to,is_current
100,Holt,CA,4111,91,2010-01-03,2010-01-03,false
100,Rio Dell,CA,4111,92,2010-01-04,2010-01-09,false
100,Windsor,CA,4111,93,2010-01-10,2010-01-10,false
100,Merced,CA,4111,94,2010-01-11,2010-01-12,false
100,Turlock,CA,4111,95,2010-01-13,2010-11-30,false
100,Woodbridge,CA,4111,96,2010-12-01,2012-07-01,false
100,Cobb,CA,4111,97,2012-07-02,null,true
